## RTOFS NetCDF 데이터를 GRIB2 JSON 포멧의 파일로 변환을 위한 분석 노트

1. 개요 및 배경

- 목표: NOAA NOMADS 서버 등에서 제공하는 RTOFS(Rapid Refresh Ocean Forecast System) 또는 HYCOM 계열의 해양 수치 모델 NetCDF 데이터를 표준 지리적 위도/경도(Rectilinear Grid) 체계에 맞게 리샘플링하고, 웹 시각화(예: deck.gl 등)에서 활용하기 쉬운 GRIB2 형태의 JSON 페이로드 구조로 변환합니다.
- 주요 해결 과제:
  - 해양 모델은 왜곡된 2차원 곡선 좌표계(Curvilinear Grid)를 사용하므로 단순 1차원 슬라이싱(ds.sel())이 불가능합니다.
  - 경도 값이 $0^\circ \sim 360^\circ$ 범위를 초과하여 누적된 경우(예: $1000^\circ$ 이상)를 전지구 표준 범위로 정규화해야 합니다.
  - 대용량 2차원 데이터에서 매 타임스텝마다 삼각화(Triangulation)를 반복하면 성능 저하가 발생하므로, 최초 1회만 Delaunay Triangulation 뼈대를 빌드하고 이후 타임스텝에서는 값만 교체(interp.values = ...)하여 최적화를 수행합니다.

2. 라이브러리 임포트 및 데이터셋 로드

**필수 라이브러리 임포트**

```python
import os
import xarray as xr
import pandas as pd
import numpy as np
import time
from scipy.interpolate import LinearNDInterpolator
import json
import gc
```

- xarray: 대용량 다차원 NetCDF/GRIB 환경 데이터셋 처리를 위한 표준 라이브러리입니다.
- scipy.interpolate.LinearNDInterpolator: 곡선 좌표계의 불규칙한 2차원 격자점(Points)을 1도 정규 격자에 선형 보간(Linear Interpolation)하기 위해 사용됩니다.
- gc: 대용량 연산 중 메모리 누수를 방지하기 위한 가비지 컬렉터 제어용입니다.

**데이터셋 오픈 (Lazy Loading 적용)**

```python
file_path="./download/rtofs_glo_uv_20261106_hcasts.nc"
ds = xr.open_dataset(file_path, chunks={"MT": 1})
```

- chunks={"MT": 1}: Dask 기반의 지연 로딩(Lazy Loading)을 활성화하여, 대용량 NetCDF 파일을 메모리에 한 번에 올리지 않고 시간 축(MT) 단위로 나누어 처리할 수 있도록 준비합니다.

3. 경도(Longitude) 범위 정규화 및 타겟 정규 격자 정의

**경도 값 모듈로(Modulo) 연산 정규화**
```python
# 정규화 전 범위 확인
ds['Longitude'].compute().values.min(), ds['Longitude'].compute().values.max()

# 360도를 초과하는 누적 경도 값을 360으로 나눈 나머지로 변환
ds['Longitude'] = xr.where(ds['Longitude'].compute() > 360, ds['Longitude'].compute() % 360, ds['Longitude'].compute())

# 정규화 후 범위 재확인 (0 ~ 360 범위로 안착)
ds['Longitude'].compute().values.min(), ds['Longitude'].compute().values.max()
```

- RTOFS 모델 특유의 누적 경도 표기법(>360°)을 표준 지구 좌표계 범위(0° ~ 360°)로 교정하여 후속 보간 시 좌표 매칭 오류를 방지합니다.

**타겟 정규 격자(Target Grid) 구성**
```python
# 1도 정규 격자 정보 정의 (모든 파일 전역 공통 격자)
target_lons = np.arange(0, 360, 1.0)
target_lats = np.arange(90, -91, -1.0)
grid_lon, grid_lat = np.meshgrid(target_lons, target_lats)
ny, nx = grid_lon.shape
```
- 전지구를 커버하는 $1^\circ \times 1^\circ$ 해상도의 정규 위경도 그리드 메쉬(grid_lon, grid_lat)를 생성합니다. 최종적으로 GRIB2 JSON이 요구하는 정형 격자 구조의 뼈대가 됩니다.


4. 시계열 축(MT) 추출 및 2차원 좌표 플래트닝

**시간 축(MT) Datetime 변환**
```python
mt_array = pd.to_datetime(ds['MT'].compute().values)
```
- 모델이 제공하는 예측/역산 타임스텝 배열을 판다스 DatetimeIndex로 변환하여 forecastTime 계산 및 파일 메타데이터에 활용합니다.

**2차원 좌표 평탄화(Flatten)**
```python
lon_2d = ds['Longitude'].compute().values.flatten()
lat_2d = ds['Latitude'].compute().values.flatten()
```
- 모델 고유의 2차원 공간 격자 형태((y, x))를 1차원 배열로 펼쳐서 삼각화 보간기의 입력 데이터 형태로 변환합니다.

5. 유효 데이터 마스킹 및 최초 삼각화(Triangulation) 빌드

**결측치(NaN) 제거 및 유효 포인트 추출**
```python
first_valid_idx = 0
u_sample = ds['u_velocity'].isel(MT=first_valid_idx, Layer=0).compute().values.flatten()
valid_mask = ~np.isnan(u_sample) & ~np.isnan(lat_2d) & ~np.isnan(lon_2d)
points = np.column_stack((lon_2d[valid_mask], lat_2d[valid_mask]))
```
- 육지 영역이나 모델 계산 영역 외곽의 결측치(NaN)를 제외한 유효 해양 데이터 포인트 좌표(points)만 정제하여 메모리 소모를 최적화합니다.


**LinearNDInterpolator 최초 뼈대 구축 (핵심 성능 최적화)**
```python
# 빌드 시간 (가장 CPU 소모가 큰 구간 중 하나)
# 임시 값으로 뼈대(Delaunay Triangulation)를 최초 1회만 빌드합니다.
t0 = time.time()
u_vals_sample = u_sample[valid_mask].astype(np.float64)
interp_u = LinearNDInterpolator(points, u_vals_sample)
v_sample = ds['v_velocity'].isel(MT=first_valid_idx, Layer=0).compute().values.flatten()
v_vals_sample = v_sample[valid_mask].astype(np.float64)
interp_v = LinearNDInterpolator(points, v_vals_sample)
print(f" 최초 삼각화 격자 기하 구조 빌드 소요 시간: {time.time() - t0:.4f}초 (총 {len(points)} 포인트)")
```
- Scipy의 LinearNDInterpolator는 내부적으로 Delaunay Triangulation을 수행하므로 최초 빌드 시 CPU 연산 비용이 가장 큽니다.

- 이 뼈대 객체(interp_u, interp_v)를 루프 외부에 미리 만들어 두고, 이후 타임스텝에서는 기하 구조를 재계산하지 않고 값만 교체하는 방식으로 고속 처리를 달성합니다.

6. 타임스텝별 루프를 통한 GRIB2 JSON 페이로드 생성


```python
mt_indices = range(0, 9)
output_dir = "../oscar"
generated_files = []

refTime = pd.to_datetime(mt_array[first_valid_idx]).strftime('%Y-%m-%dT%H:%M:%S.000Z')

# ... 루프 내부 핵심 로직 요약 ...
for mt_idx in mt_indices:
    forecastTime = pd.to_datetime(mt_array[mt_idx]).strftime('%Y-%m-%dT%H:%M:%S.000Z')
    time_str = pd.to_datetime(mt_array[mt_idx]).strftime('%Y%m%d%H')
    print(f"[{mt_idx + 1}/{mt_array.size}] MT 시간대 처리 중... ({forecastTime})")

    # [메모리 최적화] 대용량 배열 전체를 읽지 않고 해당 타임스텝 딱 '한 장'만 디스크에서 로드
    u_raw = ds['u_velocity'].isel(MT=mt_idx, Layer=0).values.flatten()
    v_raw = ds['v_velocity'].isel(MT=mt_idx, Layer=0).values.flatten()

    # float32 형식을 float64로 명시적 변환
    u_vals = u_raw[valid_mask].astype(np.float64)
    v_vals = v_raw[valid_mask].astype(np.float64)

    interp_u.values = u_vals.reshape(-1, 1)
    interp_v.values = v_vals.reshape(-1, 1)

    u_grid = interp_u(grid_lon, grid_lat)
    v_grid = interp_v(grid_lon, grid_lat)

    # NumPy의 벡터화 연산을 이용해 전체 격자 데이터를 한 번에 소수점 2자리 반올림 후 1차원 평탄화
    u_rounded = np.round(u_grid, 2).flatten()
    v_rounded = np.round(v_grid, 2).flatten()
    # 결측치(NaN) 위치를 불리언(Boolean) 마스크로 한 번에 일괄 계산
    u_nan_mask = np.isnan(u_rounded)
    v_nan_mask = np.isnan(v_rounded)

    # 미리 계산된 마스크를 활용하여 빠르게 리스트 변환 (반복문 내 무거운 연산 제거)
    u_flat = [None if isnan else float(val) for val, isnan in zip(u_rounded, u_nan_mask)]
    v_flat = [None if isnan else float(val) for val, isnan in zip(v_rounded, v_nan_mask)]

    payload = [
        {
            "header": {
                "discipline": 10,
                "disciplineName": "Oceanographic products",
                "center": -3,
                "centerName": "Earth & Space Research",
                "refTime": refTime,
                "parameterCategory": 1,
                "parameterCategoryName": "Currents",
                "parameterNumber": 2,
                "parameterNumberName": "U_component_of_current",
                "parameterUnit": "m.s-1",
                "forecastTime": int((mt_array[mt_idx] - mt_array[first_valid_idx]).total_seconds()/60/60),
                "surface1Type": 160,
                "surface1TypeName": "Depth below sea level",
                "surface1Value": 0,
                "numberPoints": len(u_flat),
                "nx": nx,
                "ny": ny,
                "lo1": float(target_lons[0]),
                "la1": float(target_lats[0]),
                "lo2": float(target_lons[-1]),
                "la2": float(target_lats[-1]),
                "dx": 1.0,
                "dy": 1.0
            },
            "data": u_flat
        },
        {
            "header": {
                "discipline": 10,
                "disciplineName": "Oceanographic products",
                "center": -3,
                "centerName": "Earth & Space Research",
                "refTime": refTime,
                "parameterCategory": 1,
                "parameterCategoryName": "Currents",
                "parameterNumber": 3,
                "parameterNumberName": "V_component_of_current",
                "parameterUnit": "m.s-1",
                "forecastTime": int((mt_array[mt_idx] - mt_array[first_valid_idx]).total_seconds()/60/60),
                "surface1Type": 160,
                "surface1TypeName": "Depth below sea level",
                "surface1Value": 0,
                "numberPoints": len(v_flat),
                "nx": nx,
                "ny": ny,
                "lo1": float(target_lons[0]),
                "la1": float(target_lats[0]),
                "lo2": float(target_lons[-1]),
                "la2": float(target_lats[-1]),
                "dx": 1.0,
                "dy": 1.0
            },
            "data": v_flat
        }
    ]

    try:
        output_filename =  f"{time_str}-surface-currents-oscar-1.0.json"
        output_filepath = os.path.join(output_dir, output_filename)
        with open(output_filepath, 'w', encoding='utf-8') as f:
            json.dump(payload, f)

        print(f"저장 완료! (용량: {os.path.getsize(output_filepath) / (1024*1024):.2f} MB)")
        # 생성된 파일명을 리스트에 추가 (상대 경로 또는 파일명만 필요시 선택)
        generated_files.append(output_filename)
    except Exception as e:
        print(f"{time_str} 파일 처리 중 에러 발생: {e}")

    # 루프 사이클마다 사용이 끝난 루프 로컬 메모리 비우기 예비 조치
    del u_raw, v_raw, u_grid, v_grid, u_flat, v_flat, u_rounded, v_rounded, u_nan_mask, v_nan_mask
    gc.collect()
    


-메모리 절약형 슬라이싱: 
    -대용량 데이터셋 전체를 로드하지 않고 isel(MT=mt_idx, Layer=0)을 통해 특정 시점의 표층(Layer=0) 유속 컴포넌트(u, v)만 한 장씩 로드합니다.
-보간기 값 교체 및 고속 연산: 
    -interp_u.values = u_vals.reshape(-1, 1) 방식을 통해 무거운 삼각화 과정을 생략하고 즉시 정규 격자에 대한 보간값(u_grid, v_grid)을 얻습니다.
-정밀도 최적화 및 JSON 변환 준비: 소수점 둘째 자리 반올림(np.round(, 2))과 NaN 값을 None으로 변환하는 리스트 컴프리헨션을 적용하여 웹용 GRIB2 JSON 포맷 스펙에 정확히 부합하는 페이로드를 완성합니다.
```

6. 파일 생성 정보 저장

```python
 # 중복 파일명 제거 (필요시 정렬 수행)
all_generated_files = []
all_generated_files.extend(generated_files)
all_generated_files = sorted(list(set(all_generated_files)))

if all_generated_files:
    # catalog.json 파일로 overwrite 저장
    catalog_path = os.path.join(output_dir, "catalog.json")
    # 저장할 구조 정의 (필요에 따라 딕셔너리 형태로 감싸거나 리스트 형태로 바로 저장 가능합니다)
    catalog_data = {
        "files": all_generated_files,
        "total_count": len(all_generated_files)
    }
    try:
        with open(catalog_path, 'w', encoding='utf-8') as f:
            json.dump(catalog_data, f, ensure_ascii=False, indent=4)

        print(f"[카탈로그 저장 완료] 총 {len(all_generated_files)}개의 파일 목록이 {catalog_path} 에 overwrite 되었습니다.")
    except Exception as e:
        print(f"카탈로그 파일 처리 중 에러 발생: {e}")
    finally:
        print("[작업 종료]")
else:
    print("[작업 종료] 파일 생성이 실패하여 catalog.json 파일을 생성하지 않았습니다.")

```

In [1]:
import os
import xarray as xr
import pandas as pd
import numpy as np
import time
from scipy.interpolate import LinearNDInterpolator
import json
import gc

In [2]:
file_path="./download/rtofs_glo_2ds_n024_prog.nc"
ds = xr.open_dataset(file_path)

In [3]:
ds

<xarray.Dataset> Size: 416MB
Dimensions:        (MT: 1, Layer: 1, Y: 3298, X: 4500)
Coordinates:
  * MT             (MT) datetime64[ns] 8B 2026-11-09
    Date           (MT) float64 8B ...
  * Layer          (Layer) int32 4B 1
  * Y              (Y) int32 13kB 1 2 3 4 5 6 ... 3293 3294 3295 3296 3297 3298
  * X              (X) int32 18kB 1 2 3 4 5 6 ... 4495 4496 4497 4498 4499 4500
    Latitude       (Y, X) float32 59MB ...
    Longitude      (Y, X) float32 59MB ...
Data variables:
    u_velocity     (MT, Layer, Y, X) float32 59MB ...
    v_velocity     (MT, Layer, Y, X) float32 59MB ...
    sst            (MT, Y, X) float32 59MB ...
    sss            (MT, Y, X) float32 59MB ...
    layer_density  (MT, Layer, Y, X) float32 59MB ...
Attributes:
    Conventions:  CF-1.0
    title:        HYCOM ATLb2.00
    institution:  National Centers for Environmental Prediction
    source:       HYCOM archive file
    experiment:   93.1
    history:      archv2ncdf2d